In [2]:
from rdkit import Chem
import numpy as np

# 1. 定义 SMARTS 模板（直接从 getCategory.py 复制即可）
dictMolPattern = {
    'Linear Carbonates': ['[*;!#1;!R][#8][#6](=O)[#8][*;!#1;!R]'],
    'Cyclic Carbonates': ['[#8]1[#6](=O)[#8][*]1','[*]1[#8][#6](=O)[#8][*]1',
                          '[*]1[*][#8][#6](=O)[#8][*]1','[*]1[*][*][#8][#6](=O)[#8][*]1'],
    'Formates': ['[#1][#6](=O)[#8][*;!#1]'],
    'Other Esters': ['[*;!#8][#6](=O)[#8][*;!#1]'],
    'Ethers': ['[*;!#1;!#8;!$([#6](=O))][#8][*;!#1;!#8;!$([#6](=O))]'],
    'Nitriles': ['[#6]#[#7]'],
    'Ketones': ['[#6][#6](=O)[#6]'],
    'Amides': ['[#6](=O)[#7]'],
    'Phosphates/Phosphine Oxides': ['[#15](=O)'],
    'Sulfones/Sulfoxides': ['[#16](=O)'],
}

dictCat2Order = {
    'Phosphates/Phosphine Oxides': 0,
    'Sulfones/Sulfoxides': 1,
    'Amides': 2,
    'Nitriles': 3,
    'Linear Carbonates': 4,
    'Cyclic Carbonates': 5,
    'Formates': 6,
    'Ketones': 7,
    'Other Esters': 8,
    'Ethers': 9
}

def assign_category(smiles: str) -> str:
    """Return main functional group category for a single SMILES."""
    try:
        mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
        if mol is None:
            return "Invalid"
    except Exception:
        return "Invalid"

    matched_cats = []
    for cat, patterns in dictMolPattern.items():
        for p in patterns:
            patt = Chem.MolFromSmarts(p)
            if patt is not None and mol.HasSubstructMatch(patt):
                matched_cats.append(cat)
                break

    if not matched_cats:
        return "Others"

    orders = [dictCat2Order[c] for c in matched_cats]
    best_idx = int(np.argmin(orders))
    return matched_cats[best_idx]

def categorize_smiles_list(smiles_list):
    """Return a list of categories for a list of SMILES."""
    return [assign_category(s) for s in smiles_list]


In [3]:
import json

with open("generated_low.json") as f:
    generated_low = json.load(f)

with open("generated_high.json") as f:
    generated_high = json.load(f)

with open("gpt4o_low_clean.json") as f:
    gpt4o_low_clean = json.load(f)

with open("gpt4o_high_clean.json") as f:
    gpt4o_high_clean = json.load(f)

with open("llama_low_clean.json") as f:
    llama_low_clean = json.load(f)

with open("llama_high_clean.json") as f:
    llama_high_clean = json.load(f)

In [4]:
mgpt_high_cat = categorize_smiles_list(generated_high)
mgpt_low_cat  = categorize_smiles_list(generated_low)
gpt4o_high_cat = categorize_smiles_list(gpt4o_high_clean)
gpt4o_low_cat  = categorize_smiles_list(gpt4o_low_clean)
llama_high_clean_cat = categorize_smiles_list(llama_high_clean)
llama_low_clean_cat = categorize_smiles_list(llama_low_clean)

[12:54:29] SMILES Parse Error: extra close parentheses while parsing: CC(C)(C)COCCNC(=O)[Au])N[Cu]
[12:54:29] SMILES Parse Error: Failed parsing SMILES 'CC(C)(C)COCCNC(=O)[Au])N[Cu]' for input: 'CC(C)(C)COCCNC(=O)[Au])N[Cu]'
[12:54:29] SMILES Parse Error: extra close parentheses while parsing: NOCCNCC(=O)[Au])NCCO[Cu]
[12:54:29] SMILES Parse Error: Failed parsing SMILES 'NOCCNCC(=O)[Au])NCCO[Cu]' for input: 'NOCCNCC(=O)[Au])NCCO[Cu]'
[12:54:29] SMILES Parse Error: extra close parentheses while parsing: COCC(CNCC=O)CCN[Cu])OC(=O)[Au]
[12:54:29] SMILES Parse Error: Failed parsing SMILES 'COCC(CNCC=O)CCN[Cu])OC(=O)[Au]' for input: 'COCC(CNCC=O)CCN[Cu])OC(=O)[Au]'
[12:54:29] SMILES Parse Error: extra open parentheses for input: 'C=CC(CO[Cu])NCC(COCC(=O)[Au]'
[12:54:29] SMILES Parse Error: extra close parentheses while parsing: COC(CO[Cu])N(C)CC(C)O[Cu])OC(=O)[Au]
[12:54:29] SMILES Parse Error: Failed parsing SMILES 'COC(CO[Cu])N(C)CC(C)O[Cu])OC(=O)[Au]' for input: 'COC(CO[Cu])N(C)CC(C)O[Cu

In [5]:
from collections import Counter

def summarize_categories(name, cats):
    total = len(cats)
    counter = Counter(cats)
    invalid = counter.get("Invalid", 0)
    valid = total - invalid

    print(f"=== {name} ===")
    print(f"Total generated: {total}")
    print(f"Valid (parsed by RDKit):   {valid}  ({valid/total:.2%})")
    print(f"Invalid (RDKit parse fail): {invalid}  ({invalid/total:.2%})")
    print()

    # 如果想顺便看看主要功能团分布（排除 Invalid）
    for cat, count in counter.items():
        if cat not in ["Invalid"]:
            print(f"{cat:30s}: {count} ({count/total:.2%})")
    print("-" * 40)

summarize_categories("minGPT High", mgpt_high_cat)
summarize_categories("minGPT Low",  mgpt_low_cat)
summarize_categories("GPT-4o High", gpt4o_high_cat)
summarize_categories("GPT-4o Low",  gpt4o_low_cat)
summarize_categories("LLaMA-3.2-3B-Instruct High", llama_high_clean_cat)
summarize_categories("LLaMA-3.2-3B-Instruct Low", llama_low_clean_cat)

=== minGPT High ===
Total generated: 100
Valid (parsed by RDKit):   96  (96.00%)
Invalid (RDKit parse fail): 4  (4.00%)

Amides                        : 33 (33.00%)
Other Esters                  : 55 (55.00%)
Ethers                        : 5 (5.00%)
Ketones                       : 1 (1.00%)
Nitriles                      : 2 (2.00%)
----------------------------------------
=== minGPT Low ===
Total generated: 100
Valid (parsed by RDKit):   80  (80.00%)
Invalid (RDKit parse fail): 20  (20.00%)

Amides                        : 44 (44.00%)
Other Esters                  : 29 (29.00%)
Sulfones/Sulfoxides           : 1 (1.00%)
Others                        : 1 (1.00%)
Nitriles                      : 3 (3.00%)
Ketones                       : 1 (1.00%)
Ethers                        : 1 (1.00%)
----------------------------------------
=== GPT-4o High ===
Total generated: 98
Valid (parsed by RDKit):   67  (68.37%)
Invalid (RDKit parse fail): 31  (31.63%)

Other Esters                  : 7 (7.14%)

In [6]:
import pandas as pd

def freq_series(categories):
    s = pd.Series(categories)
    # 去掉 Invalid，可以视情况保留或单独报告
    s = s[s != "Invalid"]
    return (s.value_counts(normalize=True) * 100).round(1)  # 百分比

df_freq = pd.DataFrame({
    "minGPT High": freq_series(mgpt_high_cat),
    "minGPT Low": freq_series(mgpt_low_cat),
    "GPT-4o High": freq_series(gpt4o_high_cat),
    "GPT-4o Low": freq_series(gpt4o_low_cat),
    "LLaMA-3.2-3B High": freq_series(llama_high_clean_cat),
    "LLaMA-3.2-3B Low": freq_series(llama_low_clean_cat),
}).fillna(0).sort_index()

df_freq


,minGPT High,minGPT Low,GPT-4o High,GPT-4o Low,LLaMA-3.2-3B High,LLaMA-3.2-3B Low
Amides,34.4,55.0,35.8,0.0,20.4,0.0
Ethers,5.2,1.2,22.4,0.0,23.7,19.4
Formates,0.0,0.0,1.5,0.0,0.0,0.0
Ketones,1.0,1.2,13.4,27.4,6.5,19.4
Nitriles,2.1,3.8,0.0,0.0,0.0,0.0
Other Esters,57.3,36.2,10.4,0.0,41.9,0.0
Others,0.0,1.2,16.4,72.6,6.5,61.2
Sulfones/Sulfoxides,0.0,1.2,0.0,0.0,1.1,0.0


In [7]:
import pandas as pd

df_full = pd.read_csv("minGPT/htp_md.csv", sep="\t")
print(df_full.head())
print(df_full["conductivity"].value_counts())

df_train = pd.read_csv("PolyGen-train-set-from-HTP-MD.csv")
print(df_train.head())
print(df_train.columns)

df_train = pd.read_csv("htpmd-trainset.csv")
print(df_train.head())
print(df_train.columns)


                         mol_smiles  conductivity
0      NC(=O)CSCC(CO[Cu])OC(=O)[Au]             1
1  CCC(F)C(=O)NC(CO[Cu])COC(=O)[Au]             0
2       CCSCCN(CCN[Cu])CCOC(=O)[Au]             1
3     C#CCN(CCOCCO[Cu])CCOC(=O)[Au]             1
4     CCC(COC(=O)[Au])C(=O)NCCO[Cu]             0
conductivity
1    5704
0    5704
Name: count, dtype: int64
   Unnamed: 0  sample_id                           mol_smiles  conductivity
0        1028       9425        COCC(CNCC(CF)OC(=O)[Au])O[Cu]      0.000076
1        4133       9426     O=C(CCNC(=O)COC(=O)[Au])NCCN[Cu]      0.000070
2        1757       9427    NC(=O)C(COC(=O)[Au])NC(=O)CCO[Cu]      0.000104
3        1984       9428  CC(COC(=O)[Au])COC(=O)C(C)(C)CO[Cu]      0.000027
4        2228       9429   COC(=O)CC(=O)NC(CO[Cu])COC(=O)[Au]      0.000038
Index(['Unnamed: 0', 'sample_id', 'mol_smiles', 'conductivity'], dtype='object')
   Unnamed: 0.1  Unnamed: 0  \
0             0           0   
1             1           1   
2          